In [ ]:
from pathlib import Path
from collections import Counter
import collections
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import networkx as nx
import numpy as np

In [ ]:
def parse_explanation_instance(explanation, setting):
    ops = explanation["operations"]

    parsed_nodes = []
    for op in ops:
        op_type = op[0]

        if setting == "delete_ops_ft":
            if op_type == "delete_node":
                parsed_nodes.append(op[1])
            elif op_type == "delete_edge":
                parsed_nodes.append(op[1][0])
                parsed_nodes.append(op[1][1])

        elif setting == "all_ops_ff":
            if op_type == "add_node":
                parsed_nodes.append(op[1])
            elif op_type == "add_edge":
                parsed_nodes.append(op[1][0])
                parsed_nodes.append(op[1][1])

        elif setting == "all_ops_ff_del":
            if op_type == "delete_node":
                parsed_nodes.append(op[1])
            elif op_type == "delete_edge":
                parsed_nodes.append(op[1][0])
                parsed_nodes.append(op[1][1])


    return parsed_nodes

In [ ]:
def load_explanation_file(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    return data

In [ ]:
def collect_node_counts(explanation_files, setting):
    influential_nodes = []
    for explanation_f in explanation_files:
        explanation_data = load_explanation_file(explanation_f)
        if setting == "both":
            temp_influential_nodes1 = parse_explanation_instance(explanation_data, 'delete_ops_ft')
            temp_influential_nodes2 = parse_explanation_instance(explanation_data, 'all_ops_ff')
            temp_influential_nodes = temp_influential_nodes1 + temp_influential_nodes2
        elif setting == "delete_ops_ft":
            temp_influential_nodes = parse_explanation_instance(explanation_data, 'delete_ops_ft')
        elif setting == "all_ops_ff":
            temp_influential_nodes = parse_explanation_instance(explanation_data, 'all_ops_ff')
        elif setting == "all_ops_ff_del":
            temp_influential_nodes = parse_explanation_instance(explanation_data, 'all_ops_ff_del')

        influential_nodes.extend(temp_influential_nodes)

    return dict(Counter(influential_nodes)), influential_nodes

In [ ]:
dataset = "2wiki"

G = nx.read_graphml(f"/home/gbalanos/GloRAG-Ex/code/KGs/lightrag/{dataset}/graph_chunk_entity_relation.graphml")

In [ ]:
operation_setting = "delete_ops_ft"
operation_setting_1 = "all_ops_ff"

explanation_dir = f"/home/gbalanos/GloRAG-Ex/code/src/counterfactuals/results/{dataset}/{operation_setting}/"
explanation_dir_1 = f"/home/gbalanos/GloRAG-Ex/code/src/counterfactuals/results/{dataset}/{operation_setting_1}/"

explanation_files = list(Path(explanation_dir).glob("*.json"))
explanation_files_1 = list(Path(explanation_dir_1).glob("*.json"))

node_counts_1, nodes_1 = collect_node_counts(explanation_files, "delete_ops_ft")
node_counts_2, nodes_2 = collect_node_counts(explanation_files_1, "all_ops_ff")

node_counts_combined = Counter(nodes_1 + nodes_2)

In [ ]:
from adjustText import adjust_text
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

def plot_graph_highlighted(G, node_freq, title="KG with Influential Nodes", figsize=(6, 4), 
                           context_hops=2, top_n=None, save_graphml=False, plot=True):
    if top_n is not None:
        node_freq = Counter(dict(node_freq.most_common(top_n)))

    influential = {n for n in node_freq if n in G}

    # ── Build local subgraph ──────────────────────────────────────────────
    context_nodes = set(influential)
    for node in influential:
        context_nodes.update(nx.single_source_shortest_path_length(G, node, cutoff=context_hops).keys())
    G = G.subgraph(context_nodes)

    if save_graphml:
        nx.write_graphml(G, f"./element_level/subgraphs/{dataset}/graph_highlighted_{dataset}.graphml")
        print(f"Saved subgraph to ./element_level/subgraphs/graph_highlighted_{dataset}.graphml ({G.number_of_nodes()} nodes, {G.number_of_edges()} edges)")

    if not plot:
        return

    fig, ax = plt.subplots(figsize=figsize, dpi=300)

    pos = nx.spring_layout(G, seed=42, k=5.0 / (G.number_of_nodes() ** 0.4), iterations=200)
    pos = {node: (x * 3, y * 5) for node, (x, y) in pos.items()}

    nx.draw_networkx_nodes(G, pos, nodelist=list(context_nodes - influential),
                           node_color="darkgray", node_size=10, ax=ax, alpha=0.9)
    nx.draw_networkx_edges(G, pos, edge_color="lightgray",
                           width=0.3, alpha=1, ax=ax, arrows=False)

    cmap     = plt.colormaps["YlOrRd"]
    freqs    = [node_freq.get(n, 0) for n in influential]
    max_freq = max(freqs) or 1
    node_colors = [cmap(f / max_freq) for f in freqs]
    node_sizes  = [10 + 30 * (f / max_freq) for f in freqs]

    nx.draw_networkx_nodes(G, pos, nodelist=list(influential),
                           node_color=node_colors, node_size=node_sizes, ax=ax, alpha=1)

    texts = []
    for node in influential:
        x, y  = pos[node]
        ratio = node_freq.get(node, 0) / max_freq
        t = ax.text(x, y, node, fontsize=7, alpha=0.6 + 0.4 * ratio,
                    ha="center", va="bottom", color="black")
        texts.append(t)

    adjust_text(texts, ax=ax)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=0, vmax=max_freq))
    sm.set_array([])

    cax = inset_axes(ax, width="30%", height="4%", loc="upper left",
                     bbox_to_anchor=(0.10, -0.15, 1, 1), bbox_transform=ax.transAxes, borderpad=0)
    cbar = fig.colorbar(sm, cax=cax, orientation="horizontal")
    cbar.set_label("Node frequency", fontsize=10, labelpad=2)
    cbar.ax.tick_params(labelsize=7)

    ax.axis("off")
    plt.tight_layout()
    plt.savefig(f"./element_level/subgraphs/both/graph_highlighted_{dataset}_new.pdf", format="pdf")
    plt.show()

In [ ]:
def degree_distribution(G):
    """Return sorted {degree: count} dict."""
    degrees = [d for _, d in G.degree()]
    return dict(sorted(Counter(degrees).items()))
 
 
def plot_degree_distribution(G, node_freq=None, title="Subgraph Degree Distribution"):
    dist = degree_distribution(G)

    fig, ax = plt.subplots(figsize=(6, 4))

    if node_freq is not None:
        deg_freq = collections.defaultdict(list)
        for node, deg in G.degree():
            deg_freq[deg].append(node_freq.get(node, 0))
        avg_freq = {k: sum(v) / len(v) for k, v in deg_freq.items()}
        max_freq = max(avg_freq.values(), default=1)
        cmap = plt.colormaps["YlGnBu"]
        colors = [cmap(avg_freq[k] / max_freq) for k in dist]
    else:
        colors = "steelblue"

    ax.bar(dist.keys(), dist.values(), color=colors, edgecolor="white")
    ax.set_xlabel("Degree $k$")
    ax.set_ylabel("Node count")
    ax.set_title(title)
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

    plt.tight_layout()
    plt.savefig(f"./feature_level/degree_dist/both_updated/degree_distribution_{dataset}.pdf", format="pdf")
    plt.show()

In [ ]:
plot_graph_highlighted(G, node_freq=node_counts_combined, top_n=30, context_hops=1, save_graphml=False, plot=True)

In [ ]:
from matplotlib import rcParams

rcParams['font.family']      = 'serif'
rcParams['font.serif']       = ['Times New Roman', 'Times', 'DejaVu Serif']
rcParams['font.size']        = 14
rcParams['axes.titlesize']   = 15
rcParams['axes.labelsize']   = 17
rcParams['xtick.labelsize']  = 15
rcParams['ytick.labelsize']  = 15
rcParams['legend.fontsize']  = 12
rcParams['lines.linewidth']  = 2.2
rcParams['lines.markersize'] = 8
rcParams['pdf.fonttype']     = 42
rcParams['ps.fonttype']      = 42

In [ ]:
def plot_combined_degree_distribution(G, explanation_files, explanation_files_1, dataset):
    _, nodes_delete = collect_node_counts(explanation_files,   "delete_ops_ft")
    _, nodes_add    = collect_node_counts(explanation_files_1, "all_ops_ff")
    # _, nodes_irr    = collect_node_counts(explanation_files_1, "all_ops_ff_del")

    def get_degrees(node_list):
        return [G.degree(n) for n in set(node_list) if n in G]

    global_degrees = [d for _, d in G.degree()]
    delete_degrees = get_degrees(nodes_delete)
    add_degrees    = get_degrees(nodes_add)
    # irr_degrees    = get_degrees(nodes_irr)        # <-- new

    # all_degrees = global_degrees + delete_degrees + add_degrees + irr_degrees
    all_degrees = global_degrees + delete_degrees + add_degrees
    min_deg = max(1, min(d for d in all_degrees if d > 0))
    max_deg = max(all_degrees)
    bins        = np.logspace(np.log10(min_deg), np.log10(max_deg), 8)
    bin_centers = 0.5 * (bins[:-1] + bins[1:])
    bin_width   = np.diff(bins)

    def get_counts_raw(degrees):
        counts, _ = np.histogram(degrees, bins=bins)
        return counts

    raw_counts = {
        "Global KG": get_counts_raw(global_degrees),
        "T→F flip":  get_counts_raw(delete_degrees),
        "F→T flip":  get_counts_raw(add_degrees),
        # "Irrelevant": get_counts_raw(irr_degrees),  # <-- new
    }

    counts = {
        label: cnt / cnt.sum() * 100 if cnt.sum() > 0 else cnt
        for label, cnt in raw_counts.items()
    }

    colors = ["#0072B2", "#E69F00", "#D55E00", "#009E73"]

    fig, ax = plt.subplots(figsize=(6, 3.8), dpi=300)

    ax.bar(bin_centers, counts["Global KG"], width=bin_width * 0.9,
           label="Global KG", color=colors[0], alpha=0.3, edgecolor="none", zorder=1)

    line_cfg = [
        ("T→F flip",   colors[1], "o"),
        ("F→T flip",   colors[2], "o"),
        # ("Irrelevant", colors[3], "o"),  # <-- new
    ]

    for label, color, marker in line_cfg:
        ax.plot(bin_centers, counts[label],
                color=color, marker=marker, markersize=5,
                linewidth=1.8, label=label, zorder=2)

    ax.set_xscale("log")
    ax.set_xlabel("Degree $k$", fontsize=19)
    ax.set_ylabel("Fraction of nodes (%)", fontsize=19)
    ax.set_ylim(0, 100)
    ax.tick_params(axis="both", labelsize=18)
    ax.legend(fontsize=16, framealpha=0.9)

    plt.tight_layout()
    plt.savefig(
        f"./feature_level/degree_dist/both_final/degree_dist_combined_{dataset}.pdf",
        format="pdf", bbox_inches="tight"
    )
    plt.show()

In [ ]:
plot_combined_degree_distribution(G, explanation_files, explanation_files_1, dataset)